In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
repo = 'https://github.com/Frederic-P/automotive_project.git'
!git clone {repo}

Cloning into 'automotive_project'...
remote: Enumerating objects: 533, done.
remote: Total 533 (delta 0), reused 0 (delta 0), pack-reused 533 (from 2)
Receiving objects: 100% (533/533), 457.31 MiB | 16.70 MiB/s, done.
Resolving deltas: 100% (253/253), done.
Updating files: 100% (61/61), done.


In [ ]:
import pandas as pd
import os
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from automotive_project.utils import cnn_helpers
import json

In [ ]:
datadir = '/content/drive/MyDrive/'
shared_dir = os.path.join(datadir, 'shared_data')  #external gdrives
testdir = '/content/testdata'
traindir = '/content/traindata'

#where to save checkpopints:
model_dest_dir = os.path.join(datadir, 'model_output', 'full_model')
os.makedirs(model_dest_dir, exist_ok=True)

for r,d,f in os.walk(shared_dir, followlinks=True):
  for file in f:
    abspath = os.path.join(r,file)
    if 'testdata' in abspath:
      base_unzipdir = testdir
    else:
      base_unzipdir = traindir
    angle = abspath.split('/')[-1].split('.')[0]
    #Yes... this will still unzip to angles - doesn't matter, and even helps with solivng an issue.
    os.makedirs(base_unzipdir, exist_ok=True)
    # -q = quiet https://linux.die.net/man/1/unzip
    !unzip -q "{abspath}" -d "{base_unzipdir}"






In [ ]:
#rebuild all data: read all csvs in traindir and testdir

def mergedf(dir):
  dfs = []
  for angle in os.listdir(dir):
    if angle.startswith('.'):
      continue
    angle_path = os.path.join(dir, angle)
    data_df = pd.read_csv(os.path.join(angle_path, 'datafile.csv'), skiprows=1, names=['brand', 'angle', 'path'])
    dfs.append(data_df)
  return pd.concat(dfs)

df_test = mergedf(testdir)
df_train = mergedf(traindir)

In [ ]:
df_train.sample(7)

,brand,angle,path
186750,nissan,rearright,/home/frederic/Downloads/traindata/rearright/o...
193988,opel,front,/home/frederic/Downloads/traindata/front/origi...
140200,lotus,rearright,/home/frederic/Downloads/traindata/rearright/a...
30755,bmw,frontleft,/home/frederic/Downloads/traindata/frontleft/o...
98441,hyundai,rearleft,/home/frederic/Downloads/traindata/rearleft/or...
85432,honda,frontleft,/home/frederic/Downloads/traindata/frontleft/a...
116721,kia,left,/home/frederic/Downloads/traindata/left/augmen...


In [ ]:
df_train.shape #awesome - let's go

(2400000, 3)

In [ ]:
def set_paths(df):
  df = df.copy()
  df['abs_path'] = '/content'+df.path.str.split('Downloads').str[-1]
  df = df.drop(columns=['path'])
  return df

def fake_cols(df, fake_columns = ['yolobox_top_left_x', 'yolobox_top_left_y', 'yolobox_bottom_right_x', 'yolobox_bottom_right_y']):
  """resnet_learner expects yolobox coords in the code - will simply provide fake
  coordinates so it's happy without making breaking changes in the code. remember that cropping was done on the PC to compress data sizes far enough to go under 15GB per angle.
  """
  df = df.copy()
  for col in fake_columns:
    df[col] = -1
  return df

In [ ]:
df_train = set_paths(df_train)
df_test = set_paths(df_test)

In [ ]:
SHAPE = 224   #required for resnet
BATCH_SIZE = 128     #how big ar teh batches for the online learning part
MAX_EPOCHS = 100    #upper limit of epochs per learning task -
                    #    NOTE THAT there is early stopping and lr plateau just as in nobteook 2.
LEARNING_RATE = 0.002  # we can start with a higher LR due to bigger batch size
#the def adam(w) lr = 0.001 - we did that with a batch 32.
#batch quadrupled, as rule of thumb you should multiply the lr with the square of the batch multiple so np.sqrt(128/32)


## VERY IMPORTANT TO REALIZE THIS: we have pre-cropped the data on a physical system
# to use as little space as possible in the cloud so I could at least fit in the
# data for angle based models... because of that we do NOT need to crop again
# however, the outcome of this notebook would be the same as running it on the
# original data, on a bare metal GPU, with CROP = TRUE This transformation was applied in the to_colab.ipynb notebook. 
CROP = False

In [ ]:
device = cnn_helpers.system_pick_device()


Using GPU for deep learning.


In [ ]:
#make an encoder:
#   LabelEncoder sorts alphabetically and since all classes are always in the traindata, these numbers
#   can be considered stable.
brands = df_train.brand.unique()
label_encoder = LabelEncoder()
label_encoder.fit(brands)

LabelEncoder()

In [ ]:
#apply label encoding on train and test dataframes.
df_train['y_encoded'] = label_encoder.transform(df_train['brand'])
df_test['y_encoded'] = label_encoder.transform(df_test['brand'])

In [ ]:
df_train = cnn_helpers.shuffle_df(df_train)
df_test = cnn_helpers.shuffle_df(df_test)

In [ ]:
df_train.head(10) #shuffled is good

,brand,angle,abs_path,y_encoded
0,bmw,rear,/content/traindata/rear/original/bmw/ad350c2a-...,3
1,mazda,rearleft,/content/traindata/rearleft/augmented/mazda/a9...,15
2,lotus,front,/content/traindata/front/augmented/lotus/6eb9c...,14
3,lotus,rearleft,/content/traindata/rearleft/augmented/lotus/1c...,14
4,lexus,rearright,/content/traindata/rearright/original/lexus/63...,13
5,renault,right,/content/traindata/right/original/renault/5d7a...,22
6,alpine,right,/content/traindata/right/augmented/alpine/a911...,1
7,suzuki,frontright,/content/traindata/frontright/augmented/suzuki...,26
8,fiat,rear,/content/traindata/rear/original/fiat/550fe57a...,6
9,jeep,front,/content/traindata/front/augmented/jeep/d6ecb1...,10


In [ ]:
X_train, y_train = cnn_helpers.get_X_y(df_train, 'y_encoded', ['brand'])
X_test, y_test = cnn_helpers.get_X_y(df_test, 'y_encoded', ['brand'])

In [ ]:
X_train = fake_cols(X_train)
X_test = fake_cols(X_test)

In [ ]:
name = f'trained model for -RESNET50- angles_mixed.keras'
model = cnn_helpers.resnet_learner(X_train, X_test, y_train, y_test, SHAPE, CROP, model_dest_dir, BATCH_SIZE, MAX_EPOCHS, LEARNING_RATE)
model.save(os.path.join(model_dest_dir, name))

CHECKPOINT found
Configuration or lr scheduler:
{
    "name": "adamw",
    "learning_rate": 0.0002500000118743628,
    "weight_decay": 0.004,
    "clipnorm": null,
    "global_clipnorm": null,
    "clipvalue": null,
    "use_ema": false,
    "ema_momentum": 0.99,
    "ema_overwrite_frequency": null,
    "loss_scale_factor": null,
    "gradient_accumulation_steps": null,
    "beta_1": 0.9,
    "beta_2": 0.999,
    "epsilon": 1e-07,
    "amsgrad": false
}
Base model layers are unfrozen. Fine-tuning the base model.
compiled model with unfrozen layers;
Epoch 1/3
18750/18750 ━━━━━━━━━━━━━━━━━━━━ 0s 582ms/step - accuracy: 0.9955 - loss: 0.0145
Epoch 1: val_accuracy improved from -inf to 0.97865, saving model to /content/drive/MyDrive/model_output/full_model/disposable_dump_of_brand_model_epoch_01__val_loss_0.0889.keras
Saving optimizer state for phase 1 - epcoh 0.
received epcoh checkpoint_epoch_0 and state 1
18750/18750 ━━━━━━━━━━━━━━━━━━━━ 13539s 718ms/step - accuracy: 0.9955 - loss: 0.014

In [ ]:
def label_encoder_to_dict(label_encoder):
  """Converts a LabelEncoder into a dictionary with integer as key and text label as value.
  ARGUMENTS:
    label_encoder: The LabelEncoder object.

  RETURNS:
    A dictionary mapping integer encoded labels with text labels as values.
  """
  d =  dict(zip(label_encoder.transform(label_encoder.classes_), label_encoder.classes_))
  d = {int(k): v for k, v in d.items()}
  return d

label_values = label_encoder_to_dict(label_encoder)


In [ ]:
json.dump(label_values, open(os.path.join(model_dest_dir, f'label_values_{angle}.json'), 'w'))

In [ ]:
print('Notebook completed, closing runtime')

Notebook completed, closing runtime


NameError: name 'runtime' is not defined